In [0]:
# =============================================================================
# Table Maintenance: Liquid Clustering & Vacuuming
# Note: Optimization is natively applied by Lakeflow on write (autoCompact/optimizeWrite).
# Use this utility for on-demand Liquid Clustering (CLUSTER BY) and storage reclamation.
# =============================================================================

# 1. Resolve environment dynamically (dev_free vs prod_azure)
current_cat = spark.sql("SELECT current_catalog()").collect()[0][0]
current_sch = spark.sql("SELECT current_schema()").collect()[0][0]

dbutils.widgets.text("catalog", current_cat, "1. Catalog Name")
dbutils.widgets.text("schema", current_sch, "2. Target Schema")

catalog = dbutils.widgets.get("catalog").strip()
schema = dbutils.widgets.get("schema").strip()
print(f"Applying Liquid Clustering maintenance for schema: {catalog}.{schema}")

# 2. Modern Liquid Clustering targets (Self-tuning, non-destructive, zero write-amplification)
tables_to_cluster = [
    ("silver_nasdaq_price_5m", "Symbol, PriceTimestamp"),
    ("silver_finnhub_news", "ArticleId, NewsDate"),
    ("fact_stock_prices_5m", "Symbol, PriceTimestamp"),
    ("fact_news_impact", "Symbol, NewsTimestamp"),
    ("dim_company", "Symbol"),
    ("dim_date", "CalendarDate")
]

for tbl_name, cluster_cols in tables_to_cluster:
    full_name = f"{catalog}.{schema}.{tbl_name}"
    try:
        if spark.catalog.tableExists(full_name):
            # Enable Liquid Clustering if not already clustered
            try:
                spark.sql(f"ALTER TABLE {full_name} CLUSTER BY ({cluster_cols})")
                print(f"[LIQUID CLUSTER] Enabled CLUSTER BY ({cluster_cols}) on {full_name}")
            except Exception:
                pass # Already clustered or managed by DLT
            
            # Run modern incremental OPTIMIZE (uses Liquid Clustering instead of Z-Order)
            print(f"[OPTIMIZE] Running incremental Liquid Cluster optimize on {full_name}...")
            spark.sql(f"OPTIMIZE {full_name}")
        else:
            print(f"[SKIP] {full_name} does not exist in active catalog/schema.")
    except Exception as e:
        print(f"[WARN] Maintenance skipped on {full_name}: {e}")

# 3. Safe Vacuuming (Retention 168 Hours / 7 Days)
vacuum_candidates = ["silver_nasdaq_price_5m", "silver_finnhub_news", "fact_stock_prices_5m", "fact_news_impact"]
for tbl in vacuum_candidates:
    full_name = f"{catalog}.{schema}.{tbl}"
    if spark.catalog.tableExists(full_name):
        try:
            print(f"[VACUUM] Reclaiming stale files for {full_name}...")
            spark.sql(f"VACUUM {full_name} RETAIN 168 HOURS")
        except Exception as e:
            print(f"[WARN] Vacuum skipped on {full_name}: {e}")

print("\n--- Liquid Clustering & maintenance routine completed. ---")


In [0]:
# catalog = dbutils.widgets.get("catalog")
# username = dbutils.widgets.get("username")
# gold_table = f"{catalog}.{username}_gold.nasdaq_news_impact_gold"

# print("--- History Log for Gold Table ---")
# display(spark.sql(f"DESCRIBE HISTORY {gold_table}"))